In [1]:
import os
import time
from pathlib import Path
from typing import List
import requests
import pandas as pd
from dotenv import load_dotenv


def load_azure_creds():
    """Load Azure Translator creds from .env, then fallback to azure key.txt."""
    load_dotenv()
    key = os.getenv("AZURE_TRANSLATOR_KEY")
    region = os.getenv("AZURE_TRANSLATOR_REGION")
    endpoint = os.getenv(
        "AZURE_TRANSLATOR_ENDPOINT",
        "https://api.cognitive.microsofttranslator.com",
    )

    key_file = Path("../../azure key.txt")
    if key_file.exists():
        for raw_line in key_file.read_text().splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("API_KEY") and not key:
                value = line.split("=", 1)[1].strip()
                key = value.strip('"').strip("'")
            if line.startswith("API_REGION") and not region:
                value = line.split("=", 1)[1].strip()
                region = value.strip('"').strip("'")

    if not key or not region:
        raise ValueError("Set AZURE_TRANSLATOR_KEY and AZURE_TRANSLATOR_REGION in .env or azure key.txt")
    return key, region, endpoint


AZURE_TRANSLATOR_KEY, AZURE_TRANSLATOR_REGION, AZURE_TRANSLATOR_ENDPOINT = load_azure_creds()

In [3]:
def azure_translate(texts: List[str], key: str, region: str, endpoint: str) -> List[str]:
    """Translate a list of texts to Filipino using Azure Translator."""
    url = f"{endpoint.rstrip('/')}/translate"
    params = {"api-version": "3.0", "from": "en", "to": "fil"}
    headers = {
        "Ocp-Apim-Subscription-Key": key,
        "Ocp-Apim-Subscription-Region": region,
        "Content-Type": "application/json",
    }
    payload = [{"Text": text} for text in texts]

    response = requests.post(url, params=params, headers=headers, json=payload)

    try:
        response_json = response.json()
    except Exception:
        print("Non-JSON response:")
        print(response.text[:500])
        raise

    if isinstance(response_json, dict) and "error" in response_json:
        # Bubble up API errors (e.g., rate limits)
        raise RuntimeError(response_json)

    try:
        translations = [item["translations"][0]["text"] for item in response_json]
        return translations
    except Exception as e:
        print("Unexpected response structure:", e)
        print(response_json)
        raise

In [4]:
def translate_in_chunks(texts: List[str], chunk_size: int = 20) -> List[str]:
    results: List[str] = []
    for start in range(0, len(texts), chunk_size):
        chunk = texts[start : start + chunk_size]
        attempt = 0
        while True:
            try:
                translated = azure_translate(chunk, AZURE_TRANSLATOR_KEY, AZURE_TRANSLATOR_REGION, AZURE_TRANSLATOR_ENDPOINT)
                break
            except RuntimeError as err:
                err_text = str(err)
                if "429" in err_text or "request limits" in err_text:
                    backoff = min(60, 5 * (attempt + 1))
                    print(f"Hit rate limit; sleeping {backoff}s then retrying (attempt {attempt + 1})...")
                    time.sleep(backoff)
                    attempt += 1
                    if attempt >= 5:
                        raise
                else:
                    raise
        results.extend(translated)
    return results

In [5]:
df_bcopa = pd.read_csv("../../datasets/cleaned/cleaned_bcopa.csv")
print(f"Loaded BCOPA rows: {len(df_bcopa)}")

Loaded BCOPA rows: 400


In [6]:
premise_list = df_bcopa["premise"].to_list()
choice1_list = df_bcopa["choice1"].to_list()
choice2_list = df_bcopa["choice2"].to_list()

print("Translating premise ...")
premise_translated = translate_in_chunks(premise_list)
print("Translating choice1 ...")
choice1_translated = translate_in_chunks(choice1_list)
print("Translating choice2 ...")
choice2_translated = translate_in_chunks(choice2_list)

Translating premise ...
Translating choice1 ...
Translating choice2 ...
Hit rate limit; sleeping 5s then retrying (attempt 1)...
Hit rate limit; sleeping 10s then retrying (attempt 2)...
Hit rate limit; sleeping 15s then retrying (attempt 3)...


In [7]:
df_bcopa["premise"] = premise_translated
df_bcopa["choice1"] = choice1_translated
df_bcopa["choice2"] = choice2_translated
df_bcopa["question"] = df_bcopa["question"].replace({"cause": "sanhi", "effect": "bunga"})

output_path = "../../datasets/translated/bing/bing_translated_bcopa.csv"
df_bcopa.to_csv(output_path, index=False)
print(f"Saved translated BCOPA to {output_path}")

Saved translated BCOPA to ../../datasets/translated/bing/bing_translated_bcopa.csv
